# 10.2 Data Aggregation

In [1]:
import pandas as pd
import numpy as np

In [10]:
import os
import sys

# 设置项目根目录（根据你的实际路径调整）
PROJECT_ROOT = '/mnt/d/quant_projects'
os.chdir(PROJECT_ROOT)
print(f"工作目录已切换到: {os.getcwd()}")

工作目录已切换到: /mnt/d/quant_projects


In [2]:
df = pd.DataFrame({'key1' : ['a', 'a', None, 'b', 'b', 'a', None],
                   'key2' : pd.Series([1, 2, 1, 2, 1, None, 1], dtype='Int64'),
                   'data1' : np.random.standard_normal(7),
                   'data2' : np.random.standard_normal(7)
                   })

In [3]:
df

,key1,key2,data1,data2
0,a,1,1.211400,-0.049662
1,a,2,0.476791,0.687318
2,NaN,1,-0.738709,0.217945
3,b,2,0.716277,0.747553
4,b,1,1.609894,0.598908
5,a,<NA>,-1.285242,-1.067361
6,NaN,1,0.474144,0.219825


In [4]:
grouped = df.groupby('key1')

In [5]:
grouped['data1'].nsmallest(2)

key1   
a     5   -1.285242
      1    0.476791
b     3    0.716277
      4    1.609894
Name: data1, dtype: float64

In [6]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [7]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.496641,1.754678
b,1,0.893617,0.148645


In [8]:
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0  0.134316  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0  1.163086  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.844095  1.211400   3.0 -0.143235  0.881074 -1.067361 -0.558511   
b     1.386490  1.609894   2.0  0.673230  0.105108  0.598908  0.636069   

                                    
           50%       75%       max  
key1                                
a    -0.049662  0.318828  0.687318  
b     0.673230  0.710391  0.747553  

[2 rows x 24 columns]

## 10.2.1 Row-by-Row Operations and Multi-Function Applications

In [11]:
tips = pd.read_csv('pydata-book/examples/tips.csv')

In [12]:
tips.head() 

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [13]:
tips['tip_pct'] = tips['tip'] / tips['total_bill']

In [14]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [15]:
grouped = tips.groupby(['day', 'smoker'])

In [16]:
grouped_pct = grouped['tip_pct']

In [17]:
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [18]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [19]:
grouped_pct.agg([('avereage', 'mean'), ('stdev', np.std)])

avereage     stdev
day  smoker                    
Fri  No      0.151650  0.024355
     Yes     0.174783  0.049553
Sat  No      0.158048  0.039323
     Yes     0.147906  0.060640
Sun  No      0.160113  0.041974
     Yes     0.187250  0.150023
Thur No      0.160298  0.038341
     Yes     0.163863  0.038213

In [20]:
functions = ['count', 'mean', 'max']

In [22]:
result = grouped[['tip_pct', 'total_bill']].agg(functions)

In [23]:
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [24]:
result['tip_pct']

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [25]:
ftuples = [('Average', 'mean'), ('Variance', np.var)]

In [27]:
grouped[['tip_pct', 'total_bill']].agg(ftuples)

tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000593  18.420000   19.197250
     Yes     0.174783  0.002456  16.813333   77.058276
Sat  No      0.158048  0.001546  19.661778   78.133210
     Yes     0.147906  0.003677  21.276667   98.973546
Sun  No      0.160113  0.001762  20.506667   64.940331
     Yes     0.187250  0.022507  24.120000  103.306779
Thur No      0.160298  0.001470  17.113111   58.300079
     Yes     0.163863  0.001460  19.190588   65.702135

In [28]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [29]:
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],
             'size' : 'sum'})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

## 10.2.2 Return aggregated data without row indexes

In [32]:
tips.groupby(['day', 'smoker'], as_index=False)[['total_bill', 'tip', 'size', 'tip_pct']].mean()

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863
